In [1]:
import os
import math
import time
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter
from problems.ThickenShell import ThickenShell
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADDomainVisualizer import CADDomainVisualizer
from Utils.CADVisualizer   import CADVisualizer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell
from Training.MainTrain import NN_Trainer, TrainingConfig
from Decoder_CLasses.ContinuousVoronoiDecoder import ContinuousVoronoiDecoder
from Decoder_CLasses.NearUniformHoneycombBaseline import NearUniformHoneycombBaseline
from Decoder_CLasses.IntrinsicSurfaceHoneycombBaseline import IntrinsicSurfaceHoneycombBaseline
from Pred_NN_Classes.ppnet import PPNet
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d, cKDTree,QhullError
from Utils.ExportAbaqus_VoxelBased import export_abaqus_voxel_fem
from Utils.ExportAbaqus import export_abaqus_shell
from Utils.embedded_centroid_stress_audit import (
    export_embedded_centroid_stress_audit,
    print_stress_case_summary,
)
from Utils.RunFEM_ExportVoxelABA import VoxelFEMAbaqusRunner
from Decoder_CLasses import Phase1VoronoiDecoder

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)

%matplotlib inline
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

Code Directory: /home/arash/HVD_SeedsBase
Test Step files Directory: /home/arash/HVD_SeedsBase/Testparts
device: cuda
PyVista backend: trame


In [2]:

def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_surface_density(fields, density_key="rho"):
    face_tensor = fields["face_tensor"]

    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary.
    if triangles.min() == 1:
        triangles = triangles - 1

    # PyVista face format:
    # [3, i, j, k, 3, i, j, k, ...]
    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    mesh = pv.PolyData(points, pv_faces)

    density = to_numpy(
        fields[density_key]
    ).reshape(-1)

    mesh.point_data["density"] = density

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": density_key,
        },
    )

    plotter.add_text(
        f"Surface density: {density_key}",
        font_size=12,
    )

    plotter.show_axes()
    plotter.show()

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv


def show_fem_distributions(
    verification,
    *,
    case_name,
    stress_limit=None,
    bins=30,
    show_edges=False,
    output_dir="FEM_Maps",
    clim_stress=None,
    clim_displacement=None,
):
    fe = verification.embedded_fem.fe
    mesh = fe.mesh

    # Stress: maximum of 8 integration points per active element
    stress = (
        fe.stress_vm_ip_active
        .detach()
        .cpu()
        .numpy()
        .max(axis=1)
    )
    element_ids = np.asarray(mesh.active_element_ids, dtype=np.int64)

    # Displacement magnitude: one value per mesh node
    u = (
        fe.u.detach()
        .cpu()
        .numpy()
        .reshape(mesh.numNodes, 3)
    )
    displacement = np.linalg.norm(u, axis=1)
    active_node_ids = np.asarray(mesh.active_node_ids, dtype=np.int64)

    if len(stress) != len(element_ids):
        raise ValueError("Stress values do not match the active elements.")

    # Build a grid containing active C3D8 elements only
    connectivity = mesh.elemNodes[element_ids].astype(np.int64)
    cells = np.column_stack([
        np.full(len(element_ids), 8, dtype=np.int64),
        connectivity,
    ]).ravel()

    grid = pv.UnstructuredGrid(
        cells,
        np.full(len(element_ids), pv.CellType.HEXAHEDRON, dtype=np.uint8),
        np.asarray(mesh.nodeXYZ),
    )

    stress_name = "Max IP von Mises (MPa)"
    displacement_name = "Displacement magnitude (mm)"

    grid.cell_data[stress_name] = stress
    grid.point_data[displacement_name] = displacement
    surface = grid.extract_surface()

    # Two maps, with the same camera position
    plotter = pv.Plotter(shape=(1, 2), window_size=(1600, 700))

    Clim_stress_Def = None

    stress_range = (
        (0.0, float(stress.max()))
        if clim_stress is None
        else tuple(clim_stress)
    )

    displacement_range = (
        (0.0, float(displacement[active_node_ids].max()))
        if clim_displacement is None
        else tuple(clim_displacement)
    )

    plotter.subplot(0, 0)
    plotter.add_mesh(
        surface,
        scalars=stress_name,
        preference="cell",
        cmap="turbo",
        clim=stress_range,
        show_edges=show_edges,
        scalar_bar_args={"title": stress_name},
    )
    plotter.add_text(f"{case_name}: stress", font_size=11)
    plotter.add_axes()

    plotter.subplot(0, 1)
    plotter.add_mesh(
        surface,
        scalars=displacement_name,
        preference="point",
        cmap="viridis",
        clim=displacement_range,
        show_edges=show_edges,
        scalar_bar_args={"title": displacement_name},
    )
    plotter.add_text(f"{case_name}: displacement", font_size=11)
    plotter.add_axes()

    plotter.link_views()
    plotter.reset_camera()

    # Histograms: one count per active element and per active node
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].hist(stress, bins=bins)
    if stress_limit is not None:
        axes[0].axvline(
            stress_limit,
            color="red",
            linestyle="--",
            label=f"Limit: {stress_limit:g} MPa",
        )
        axes[0].legend()
    axes[0].set_xlabel(stress_name)
    axes[0].set_ylabel("Number of active elements")
    axes[0].set_title("Element stress distribution")

    axes[1].hist(displacement[active_node_ids], bins=bins)
    axes[1].set_xlabel(displacement_name)
    axes[1].set_ylabel("Number of active nodes")
    axes[1].set_title("Nodal displacement distribution")

    fig.suptitle(case_name)
    fig.tight_layout()

    # Save each model under its own name
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    grid.save(output_dir / f"{case_name}_fields.vtu")
    fig.savefig(output_dir / f"{case_name}_histograms.png", dpi=200)

    plotter.show(screenshot=str(output_dir / f"{case_name}_maps.png"))
    plt.show()

    worst_element_rank = int(np.argmax(stress))
    worst_node = int(active_node_ids[np.argmax(displacement[active_node_ids])])

    print(f"Maximum stress: {stress.max():.4f} MPa")
    print(f"  Element centre: {mesh.elemCenters[element_ids[worst_element_rank]]}")
    print(f"Maximum displacement: {displacement[worst_node]:.6f} mm")
    print(f"  Node position: {mesh.nodeXYZ[worst_node]}")

    if stress_limit is not None:
        above = int(np.count_nonzero(stress > stress_limit))
        print(
            f"Elements above {stress_limit:g} MPa: "
            f"{above}/{len(stress)} ({100 * above / len(stress):.2f}%)"
        )

    return grid

In [3]:
def show_two_surface_densities(
    face_tensor,
    density_phase1,
    density_phase2,
    *,
    title_phase1="Phase 1 - Continuous",
    title_phase2="Phase 2 - Graph",
    binary=False,
    threshold=0.5,
):
    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary
    if triangles.min() == 1:
        triangles = triangles - 1

    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    # --------------------------------------------------------
    # Densities
    # --------------------------------------------------------

    rho1 = to_numpy(density_phase1).reshape(-1)
    rho2 = to_numpy(density_phase2).reshape(-1)

    if len(rho1) != len(points):
        raise ValueError(
            f"Phase-1 density has {len(rho1)} values "
            f"but surface has {len(points)} vertices."
        )

    if len(rho2) != len(points):
        raise ValueError(
            f"Phase-2 density has {len(rho2)} values "
            f"but surface has {len(points)} vertices."
        )

    if binary:
        rho1 = (rho1 > threshold).astype(float)
        rho2 = (rho2 > threshold).astype(float)

    # --------------------------------------------------------
    # Separate meshes so the scalar fields do not overwrite
    # each other
    # --------------------------------------------------------

    mesh1 = pv.PolyData(points, pv_faces)
    mesh2 = pv.PolyData(points, pv_faces)

    mesh1.point_data["density"] = rho1
    mesh2.point_data["density"] = rho2

    # --------------------------------------------------------
    # Side-by-side visualization
    # --------------------------------------------------------

    plotter = pv.Plotter(
        shape=(1, 2),
        window_size=(1700, 750),
    )

    # Phase 1
    plotter.subplot(0, 0)

    plotter.add_mesh(
        mesh1,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": "Density",
        },
    )

    plotter.add_text(
        title_phase1,
        font_size=12,
    )

    plotter.add_axes()

    # Phase 2
    plotter.subplot(0, 1)

    plotter.add_mesh(
        mesh2,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": "Density",
        },
    )

    plotter.add_text(
        title_phase2,
        font_size=12,
    )

    plotter.add_axes()

    # Same camera for both
    plotter.link_views()
    plotter.reset_camera()

    plotter.show()

    return mesh1, mesh2

In [4]:
viz = CADVisualizer()
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Planar = TesPartsDir / "Planar.stp"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
FreeSharp2 = TesPartsDir / "FreeSharp2.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Helmet = TesPartsDir / "Helmet_New.stp"
FreeForm_Bend = TesPartsDir / "FreeForm_bend.stp"
CircBracket_Half = TesPartsDir / "CircBracket_Half.stp"
HelmetNew_Half = TesPartsDir / "HelmetNew_Half.stp"
Wing_NACA2412 = TesPartsDir / "Wing_NACA2412.stp"

aircraft_radome = TesPartsDir / "aircraft_radome.stp"

shape_path = Planar
Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=512,
    mesh_size_scale = 0.4,
)

cad_domain, face_mesh = Face_Cad.generate_from_file(shape_path)
dx,dy,dz = Face_Cad.print_face_info();

points = face_mesh["points_xyz"].detach().cpu().numpy()
faces = face_mesh["pv_faces"].detach().cpu().numpy()

mesh = pv.PolyData(points, faces)

plotter = pv.Plotter()
plotter.add_mesh(
    mesh,
    show_edges=True,
)
plotter.add_axes()
plotter.show()

points_uv  = face_mesh["uv"]
points_xyz = face_mesh["points_xyz"]
Xu         = face_mesh["Xu"]
Xv         = face_mesh["Xv"]
faces_ijk = face_mesh["faces_ijk"].long()

p0 = points_xyz[faces_ijk[:, 0]]
p1 = points_xyz[faces_ijk[:, 1]]
p2 = points_xyz[faces_ijk[:, 2]]

face_areas = 0.5 * torch.linalg.vector_norm(
    torch.cross(
        p1 - p0,
        p2 - p0,
        dim=1,
    ),
    dim=1,
)

surface_area_weights = torch.zeros(
    points_xyz.shape[0],
    device=points_xyz.device,
    dtype=points_xyz.dtype,
)

third_area = face_areas / 3.0

for lv in range(3):
    surface_area_weights.scatter_add_(
        0,
        faces_ijk[:, lv],
        third_area,
    )

print(
    "Surface area:",
    float(surface_area_weights.sum())
)

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    :  - Label 'Shapes/Document' (2D)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (BSpline)
Info    : [ 30%] Meshing curve 2 (BSpline)
Info    : [ 60%] Meshing curve 3 (BSpline)
Info    : [ 80%] Meshing curve 4 (BSpline)
Info    : Done meshing 1D (Wall 0.000456419s, CPU 0s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.34234s, CPU 0.331896s)
Info    : 20075 nodes 40152 elements
Info    : Clearing all models and views...
Info    : Done clearing all models and views

=== Active Face Info ===
XYZ bounds:
  X: [-0.000000, 10.000000]  span=10.000000
  Y: [-0.000000, 10.000000]  span=10.000000
  Z: [-0.000000, 0.000000]  span=0.000000

UV bounds:
  U: [-10.000000, 0.000000]
  V: [0.000000, 10.000000]

Periodic:
  U periodic: False
  V periodic: False



Widget(value='<iframe src="http://localhost:39739/index.html?ui=P_0x7aef2d2783d0_0&reconnect=auto" class="pyvi…

Surface area: 100.00000762939453


In [5]:
import torch
import numpy as np
import pyvista as pv

from Utils.DifferentiableFilters import smooth_heaviside_projection


def compare_phase1_phase2(
    case_name,
    seeds_uv,
    *,
    tau,

    # ------------------------------------------------------------
    # Geometry
    # ------------------------------------------------------------
    strut_thickness=0.20,
    tube_beta=0.02,

    # ------------------------------------------------------------
    # Phase-1 seed/activity controls
    # ------------------------------------------------------------
    duplicate_merge_sigma=0.02,

    # ------------------------------------------------------------
    # Density comparison
    # ------------------------------------------------------------
    rho_threshold=0.50,

    # ------------------------------------------------------------
    # Common FEM projection
    # ------------------------------------------------------------
    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,

    # ------------------------------------------------------------
    # Phase-1 boundary attachment
    # ------------------------------------------------------------
    boundary_attach_width=2e-5,
    boundary_attach_beta=1e-5,
    boundary_attach_alpha=1.0,

    # ------------------------------------------------------------
    # Fibre visualization
    # ------------------------------------------------------------
    fiber_density_threshold=0.50,
    max_fiber_arrows=700,
    fiber_arrow_scale=0.025,

    visualize=True,
):

    # ============================================================
    # Common geometry
    # ============================================================

    seeds_uv = seeds_uv.to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    S = int(seeds_uv.shape[0])

    strut_thickness = float(strut_thickness)

    # Phase 1 takes radius / half-width.
    strut_radius = 0.5 * strut_thickness

    print("\n" + "=" * 78)
    print(f"CASE: {case_name}")
    print(f"Seeds                 : {S}")
    print(f"Full strut thickness  : {strut_thickness:.6f}")
    print(f"Strut radius          : {strut_radius:.6f}")
    print("=" * 78)

    # ============================================================
    # PHASE 1
    # Implicit continuous Voronoi decoder
    # ============================================================

    phase1_decoder = Phase1VoronoiDecoder(

        n_seeds=S,

        use_Metric_anisotropy=False,

        # P1 expects radius, not full width.
        fixed_strut_radius=strut_radius,

        # Legacy output requirement only.
        # Not used to define in-plane strut width.
        fixed_height=1.0,

        physical_tube_beta=float(
            tube_beta
        ),

        duplicate_merge_sigma=float(
            duplicate_merge_sigma
        ),

        territory_min_ratio=0.05,

        continuous_length_calibration=1.0,

        use_boundary_attachment=True,

        point_chunk_size=1024,

        boundary_attach_width=float(
            boundary_attach_width
        ),

        boundary_attach_beta=float(
            boundary_attach_beta
        ),

        boundary_attach_alpha=float(
            boundary_attach_alpha
        ),

        # No internal projection.
        density_projection_strength=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # Required by P1 API.
    # Does not control radius because fixed_strut_radius is set.
    w_raw = torch.zeros(
        S,
        S,
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # PHASE 2
    # Explicit Voronoi graph decoder
    # ============================================================

    phase2_decoder = ContinuousVoronoiDecoder(

        Cad_domain=Face_Cad,

        face_mesh=face_mesh,

        n_seeds=S,

        # P2 expects FULL width.
        strut_thickness=strut_thickness,

        beta=float(
            tube_beta
        ),

        tube_curve_samples=64,

        tube_distance_tau=float(
            tube_beta
        ),

        tube_density_tau=float(
            tube_beta
        ),

        tube_fiber_tau=float(
            tube_beta
        ),

        rho_min=0.0,

    ).to(
        device=points_uv.device,
        dtype=points_uv.dtype,
    )

    # ============================================================
    # Evaluate
    # ============================================================

    with torch.no_grad():

        out_p1 = phase1_decoder(

            points_uv=points_uv,

            Xu=Xu,

            Xv=Xv,

            tau=float(
                tau
            ),

            seeds_raw=seeds_uv,

            w_raw=w_raw,

            h_raw=None,

            points_face_id=points_face_id,

            boundary_uv=boundary_uv,

            boundary_face_id=boundary_face_id,

            boundary_curve_offsets=boundary_curve_offsets,

            surface_area_weights=surface_area_weights,
        )

        out_p2 = phase2_decoder(

            seeds_uv=seeds_uv,

            generate_density_fiber=True,
        )

    # ============================================================
    # Raw density + fibre
    # ============================================================

    rho1_raw = out_p1["rho"]

    rho2_raw = out_p2["rho"]

    fiber1 = out_p1["fiber3d"]

    fiber2 = out_p2["fiber3d"]

    # Normalize fibre directions.
    fiber1 = torch.nn.functional.normalize(
        fiber1,
        dim=-1,
        eps=1e-8,
    )

    fiber2 = torch.nn.functional.normalize(
        fiber2,
        dim=-1,
        eps=1e-8,
    )

    # ============================================================
    # Common FEM-style density projection
    # ============================================================

    rho1_proj = smooth_heaviside_projection(

        rho1_raw,

        beta=float(
            projection_beta
        ),

        eta=float(
            projection_eta
        ),

        strength=float(
            projection_strength
        ),

        eps=1e-8,

        debug=False,
    )

    rho2_proj = smooth_heaviside_projection(

        rho2_raw,

        beta=float(
            projection_beta
        ),

        eta=float(
            projection_eta
        ),

        strength=float(
            projection_strength
        ),

        eps=1e-8,

        debug=False,
    )

    # ============================================================
    # Binary fields
    # Only for geometry comparison / visualization
    # ============================================================

    rho1_bin = (
        rho1_proj >= rho_threshold
    ).to(
        rho1_proj.dtype
    )

    rho2_bin = (
        rho2_proj >= rho_threshold
    ).to(
        rho2_proj.dtype
    )

    # ============================================================
    # Nominal thickness diagnostics
    # ============================================================

    print("\n--- Nominal thickness ---")

    print(
        "Requested full thickness :",
        strut_thickness,
    )

    print(
        "P1 imposed radius        :",
        strut_radius,
    )

    print(
        "P1 implied full thickness:",
        2.0 * strut_radius,
    )

    print(
        "P2 reported radius       :",
        float(
            out_p2[
                "centerline_radius"
            ]
        ),
    )

    print(
        "P2 full thickness        :",
        float(
            out_p2[
                "strut_thickness"
            ]
        ),
    )

    # ============================================================
    # Density ranges
    # ============================================================

    def print_density_range(
        name,
        rho,
    ):

        print(
            f"{name:<14}: "
            f"min={float(rho.min()):.6f}  "
            f"mean={float(rho.mean()):.6f}  "
            f"max={float(rho.max()):.6f}"
        )

    print(
        "\n--- Density ranges ---"
    )

    print_density_range(
        "P1 raw",
        rho1_raw,
    )

    print_density_range(
        "P2 raw",
        rho2_raw,
    )

    print_density_range(
        "P1 projected",
        rho1_proj,
    )

    print_density_range(
        "P2 projected",
        rho2_proj,
    )

    # ============================================================
    # LENGTH
    # ============================================================

    L1 = out_p1[
        "continuous_voronoi_length"
    ]

    curves_xyz = out_p2[
        "edge_curves_xyz"
    ]

    segment_vectors = (
        curves_xyz[:, 1:, :]
        -
        curves_xyz[:, :-1, :]
    )

    segment_lengths = (
        torch.linalg.vector_norm(
            segment_vectors,
            dim=-1,
        )
    )

    edge_lengths = (
        segment_lengths.sum(
            dim=1
        )
    )

    graph = out_p2[
        "graph"
    ]

    edge_type = graph[
        "edge_type"
    ]

    # Type 4 = domain shell boundary.
    internal_mask = (
        edge_type != 4
    )

    boundary_mask = (
        edge_type == 4
    )

    L2_internal = (
        edge_lengths[
            internal_mask
        ].sum()
    )

    L2_boundary = (
        edge_lengths[
            boundary_mask
        ].sum()
    )

    L2_total = (
        edge_lengths.sum()
    )

    length_error_pct = (

        100.0

        * torch.abs(
            L1
            -
            L2_internal
        )

        / L2_internal
        .abs()
        .clamp_min(
            1e-8
        )
    )

    print(
        "\n--- Length ---"
    )

    print(
        f"P1 continuous Voronoi : "
        f"{float(L1):.6f}"
    )

    print(
        f"P2 internal Voronoi   : "
        f"{float(L2_internal):.6f}"
    )

    print(
        f"P2 boundary shell     : "
        f"{float(L2_boundary):.6f}"
    )

    print(
        f"P2 total              : "
        f"{float(L2_total):.6f}"
    )

    print(
        f"P1/P2 internal error  : "
        f"{float(length_error_pct):.3f} %"
    )

    # ============================================================
    # Material area helper
    # ============================================================

    thresholds = [
        0.01,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
    ]

    def material_area(
        rho,
        threshold,
    ):

        return (

            (
                rho >= threshold
            )
            .to(
                surface_area_weights.dtype
            )

            * surface_area_weights

        ).sum()

    # ============================================================
    # RAW AREA SWEEP
    # ============================================================

    print(
        "\n--- Raw density area sweep ---"
    )

    for thr in thresholds:

        A1 = material_area(
            rho1_raw,
            thr,
        )

        A2 = material_area(
            rho2_raw,
            thr,
        )

        if float(A2) > 1e-8:

            ratio_text = (
                f"{float(A1 / A2):.4f}"
            )

        else:

            ratio_text = "N/A"

        print(
            f"rho >= {thr:4.2f}: "
            f"P1={float(A1):9.4f}   "
            f"P2={float(A2):9.4f}   "
            f"ratio={ratio_text}"
        )

    # ============================================================
    # PROJECTED AREA SWEEP
    # ============================================================

    print(
        "\n--- FEM-projected area sweep ---"
    )

    for thr in thresholds:

        A1 = material_area(
            rho1_proj,
            thr,
        )

        A2 = material_area(
            rho2_proj,
            thr,
        )

        if float(A2) > 1e-8:

            ratio_text = (
                f"{float(A1 / A2):.4f}"
            )

        else:

            ratio_text = "N/A"

        print(
            f"rho >= {thr:4.2f}: "
            f"P1={float(A1):9.4f}   "
            f"P2={float(A2):9.4f}   "
            f"ratio={ratio_text}"
        )

    # ============================================================
    # Projected density errors
    # ============================================================

    rho_diff = (
        rho1_proj
        -
        rho2_proj
    )

    rho_mae = (
        rho_diff
        .abs()
        .mean()
    )

    rho_rmse = torch.sqrt(
        rho_diff
        .pow(2)
        .mean()
    )

    weighted_rho_mae = (

        (
            rho_diff.abs()
            *
            surface_area_weights
        ).sum()

        /

        surface_area_weights
        .sum()
        .clamp_min(
            1e-8
        )
    )

    print(
        "\n--- FEM-projected density error ---"
    )

    print(
        "MAE          :",
        float(rho_mae),
    )

    print(
        "RMSE         :",
        float(rho_rmse),
    )

    print(
        "Weighted MAE :",
        float(weighted_rho_mae),
    )

    # ============================================================
    # Binary geometry comparison
    # ============================================================

    A1_bin = (
        rho1_bin
        *
        surface_area_weights
    ).sum()

    A2_bin = (
        rho2_bin
        *
        surface_area_weights
    ).sum()

    binary_area_ratio = (
        A1_bin
        /
        A2_bin.clamp_min(
            1e-8
        )
    )

    print(
        "\n--- Final binary geometry @ rho=0.5 ---"
    )

    print(
        "P1 binary area:",
        float(A1_bin),
    )

    print(
        "P2 binary area:",
        float(A2_bin),
    )

    print(
        "P1/P2 ratio :",
        float(binary_area_ratio),
    )

    # ============================================================
    # Fibre comparison
    #
    # Fibre is axial:
    #
    #       t == -t
    #
    # therefore use |dot|.
    # ============================================================

    axial_dot = torch.abs(

        (
            fiber1
            *
            fiber2
        ).sum(
            dim=-1
        )

    ).clamp(
        0.0,
        1.0,
    )

    angle_deg = (

        torch.acos(
            axial_dot
        )

        * 180.0

        / torch.pi
    )

    # Compare only where BOTH projected
    # representations contain material.
    common_material = (

        (
            rho1_proj
            >= rho_threshold
        )

        &

        (
            rho2_proj
            >= rho_threshold
        )
    )

    if bool(
        common_material.any()
    ):

        common_angles = (
            angle_deg[
                common_material
            ]
        )

        fiber_angle_mean = (
            common_angles.mean()
        )

        fiber_angle_max = (
            common_angles.max()
        )

        fiber_angle_p95 = torch.quantile(
            common_angles,
            0.95,
        )

    else:

        fiber_angle_mean = (
            angle_deg.new_tensor(
                float("nan")
            )
        )

        fiber_angle_max = (
            angle_deg.new_tensor(
                float("nan")
            )
        )

        fiber_angle_p95 = (
            angle_deg.new_tensor(
                float("nan")
            )
        )

    print(
        "\n--- Fibre field ---"
    )

    print(
        "Mean axial angle error:",
        float(
            fiber_angle_mean
        ),
        "deg",
    )

    print(
        "95th percentile error :",
        float(
            fiber_angle_p95
        ),
        "deg",
    )

    print(
        "Max axial angle error :",
        float(
            fiber_angle_max
        ),
        "deg",
    )

    # ============================================================
    # Topology
    # ============================================================

    phase2_seed_mask = (
        out_p1.get(
            "phase2_seed_mask",
            None,
        )
    )

    if (
        phase2_seed_mask
        is not None
    ):

        handoff_count = int(

            phase2_seed_mask
            .sum()
            .detach()
            .cpu()
        )

    else:

        handoff_count = S

    print(
        "\n--- Topology ---"
    )

    print(
        "Original seeds      :",
        S,
    )

    print(
        "P1 handoff seeds    :",
        handoff_count,
    )

    print(
        "P2 graph edge count :",
        int(
            curves_xyz.shape[0]
        ),
    )

    # ============================================================
    # Density parameters
    # ============================================================

    local_scale = (

        phase2_decoder
        ._local_uv_to_xyz_scale(

            Xu,
            Xv,
            points_xyz,
        )
    )

    p2_tau_density = (

        phase2_decoder
        .tube_density_tau

        *
        local_scale
    )

    print(
        "\n--- Density parameters ---"
    )

    print(
        "P1 physical radius:",
        strut_radius,
    )

    print(
        "P1 physical beta  :",
        float(
            phase1_decoder
            .physical_tube_beta
        ),
    )

    print(
        "P2 radius         :",
        float(
            out_p2[
                "centerline_radius"
            ]
        ),
    )

    print(
        "P2 local scale    :",
        float(
            local_scale
        ),
    )

    print(
        "P2 effective beta :",
        float(
            p2_tau_density
        ),
    )

    # ============================================================
    # VISUALIZATION
    # ============================================================

    if visualize:

        xyz_np = (
            points_xyz
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        tri_np = (
            faces_ijk
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        n_faces = (
            tri_np.shape[0]
        )

        pv_faces = np.hstack(
            [
                np.full(
                    (
                        n_faces,
                        1,
                    ),
                    3,
                    dtype=np.int64,
                ),

                tri_np,
            ]
        ).reshape(-1)

        # --------------------------------------------------------
        # Convert fields
        # --------------------------------------------------------

        rho1_raw_np = (
            rho1_raw
            .detach()
            .cpu()
            .numpy()
        )

        rho2_raw_np = (
            rho2_raw
            .detach()
            .cpu()
            .numpy()
        )

        rho1_proj_np = (
            rho1_proj
            .detach()
            .cpu()
            .numpy()
        )

        rho2_proj_np = (
            rho2_proj
            .detach()
            .cpu()
            .numpy()
        )

        rho1_bin_np = (
            rho1_bin
            .detach()
            .cpu()
            .numpy()
        )

        rho2_bin_np = (
            rho2_bin
            .detach()
            .cpu()
            .numpy()
        )

        fiber1_np = (
            fiber1
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        fiber2_np = (
            fiber2
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        # ========================================================
        # Helper: build mesh
        # ========================================================

        def make_mesh(
            scalar_field,
        ):

            mesh = pv.PolyData(
                xyz_np,
                pv_faces,
            )

            mesh["rho"] = (
                scalar_field.astype(
                    np.float32
                )
            )

            return mesh

        # ========================================================
        # Helper: add fibre arrows
        # ========================================================

        def add_fiber_vectors(
            plotter,
            density,
            fiber_vectors,
            *,
            threshold,
            max_arrows,
            scale_factor,
        ):

            density = np.asarray(
                density
            )

            fiber_vectors = np.asarray(
                fiber_vectors
            )

            fiber_norm = np.linalg.norm(
                fiber_vectors,
                axis=1,
            )

            valid = (

                np.isfinite(
                    density
                )

                &

                np.isfinite(
                    fiber_vectors
                ).all(
                    axis=1
                )

                &

                (
                    density
                    >= threshold
                )

                &

                (
                    fiber_norm
                    > 1e-10
                )
            )

            idx = np.flatnonzero(
                valid
            )

            if idx.size == 0:

                return 0

            # --------------------------------------------
            # Subsample arrows for readability
            # --------------------------------------------

            if idx.size > max_arrows:

                stride = int(
                    np.ceil(
                        idx.size
                        /
                        max_arrows
                    )
                )

                idx = idx[
                    ::stride
                ]

            arrow_points = (
                xyz_np[
                    idx
                ]
            )

            arrow_vectors = (
                fiber_vectors[
                    idx
                ]
            )

            # Normalize once more before glyphing.
            arrow_vectors = (

                arrow_vectors

                /

                np.linalg.norm(
                    arrow_vectors,
                    axis=1,
                    keepdims=True,
                )
            )

            # --------------------------------------------
            # Physical arrow size relative to model
            # --------------------------------------------

            bbox = (
                xyz_np.max(
                    axis=0
                )
                -
                xyz_np.min(
                    axis=0
                )
            )

            diagonal = np.linalg.norm(
                bbox
            )

            arrow_length = (
                scale_factor
                *
                diagonal
            )

            cloud = pv.PolyData(
                arrow_points
            )

            cloud[
                "fiber"
            ] = arrow_vectors

            glyphs = cloud.glyph(

                orient="fiber",

                scale=False,

                factor=float(
                    arrow_length
                ),

                geom=pv.Arrow(),
            )

            plotter.add_mesh(
                glyphs,
            )

            return int(
                idx.size
            )

        # ========================================================
        # 3D comparison
        #
        # Row 1: raw densities
        # Row 2: projected densities + fibre vectors
        # Row 3: binary geometries
        # ========================================================

        plotter = pv.Plotter(

            shape=(3, 2),

            window_size=(
                1500,
                1500,
            ),
        )

        # --------------------------------------------------------
        # P1 RAW
        # --------------------------------------------------------

        plotter.subplot(
            0,
            0,
        )

        mesh = make_mesh(
            rho1_raw_np
        )

        plotter.add_text(
            "Phase 1 — raw density",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P2 RAW
        # --------------------------------------------------------

        plotter.subplot(
            0,
            1,
        )

        mesh = make_mesh(
            rho2_raw_np
        )

        plotter.add_text(
            "Phase 2 — raw density",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P1 PROJECTED + FIBRES
        # --------------------------------------------------------

        plotter.subplot(
            1,
            0,
        )

        mesh = make_mesh(
            rho1_proj_np
        )

        plotter.add_text(
            "Phase 1 — FEM projected + fibres",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            opacity=0.75,

            show_edges=False,
        )

        p1_arrow_count = add_fiber_vectors(

            plotter,

            rho1_proj_np,

            fiber1_np,

            threshold=float(
                fiber_density_threshold
            ),

            max_arrows=int(
                max_fiber_arrows
            ),

            scale_factor=float(
                fiber_arrow_scale
            ),
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P2 PROJECTED + FIBRES
        # --------------------------------------------------------

        plotter.subplot(
            1,
            1,
        )

        mesh = make_mesh(
            rho2_proj_np
        )

        plotter.add_text(
            "Phase 2 — FEM projected + fibres",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            opacity=0.75,

            show_edges=False,
        )

        p2_arrow_count = add_fiber_vectors(

            plotter,

            rho2_proj_np,

            fiber2_np,

            threshold=float(
                fiber_density_threshold
            ),

            max_arrows=int(
                max_fiber_arrows
            ),

            scale_factor=float(
                fiber_arrow_scale
            ),
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P1 BINARY
        # --------------------------------------------------------

        plotter.subplot(
            2,
            0,
        )

        mesh = make_mesh(
            rho1_bin_np
        )

        plotter.add_text(
            "Phase 1 — binary geometry",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # P2 BINARY
        # --------------------------------------------------------

        plotter.subplot(
            2,
            1,
        )

        mesh = make_mesh(
            rho2_bin_np
        )

        plotter.add_text(
            "Phase 2 — binary geometry",
            font_size=11,
        )

        plotter.add_mesh(

            mesh,

            scalars="rho",

            clim=[
                0.0,
                1.0,
            ],

            show_edges=False,
        )

        plotter.add_axes()

        plotter.camera_position = (
            "iso"
        )

        # --------------------------------------------------------
        # Same view everywhere
        # --------------------------------------------------------

        plotter.link_views()

        plotter.show()

        print(
            "\n--- Fibre visualization ---"
        )

        print(
            "P1 arrows shown:",
            p1_arrow_count,
        )

        print(
            "P2 arrows shown:",
            p2_arrow_count,
        )

        print(
            "Fibre density threshold:",
            fiber_density_threshold,
        )

    # ============================================================
    # Return metrics
    # ============================================================

    metrics = {

        "case":
            case_name,

        "seed_count":
            S,

        "strut_thickness":
            strut_thickness,

        "strut_radius":
            strut_radius,

        "P1_length":
            float(
                L1
            ),

        "P2_internal_length":
            float(
                L2_internal
            ),

        "P2_boundary_length":
            float(
                L2_boundary
            ),

        "P2_total_length":
            float(
                L2_total
            ),

        "length_error_pct":
            float(
                length_error_pct
            ),

        "projected_rho_mae":
            float(
                rho_mae
            ),

        "projected_rho_rmse":
            float(
                rho_rmse
            ),

        "projected_weighted_rho_mae":
            float(
                weighted_rho_mae
            ),

        "P1_binary_area":
            float(
                A1_bin
            ),

        "P2_binary_area":
            float(
                A2_bin
            ),

        "binary_area_ratio":
            float(
                binary_area_ratio
            ),

        "fiber_angle_mean_deg":
            float(
                fiber_angle_mean
            ),

        "fiber_angle_p95_deg":
            float(
                fiber_angle_p95
            ),

        "fiber_angle_max_deg":
            float(
                fiber_angle_max
            ),

        "P1_handoff_seeds":
            handoff_count,

        "P2_edge_count":
            int(
                curves_xyz.shape[0]
            ),
    }

    return (
        metrics,
        out_p1,
        out_p2,
    )

In [6]:
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

In [7]:
# ============================================================
# Common CAD data
# ============================================================

points_uv  = face_mesh["uv"]
points_xyz = face_mesh["points_xyz"]
Xu         = face_mesh["Xu"]
Xv         = face_mesh["Xv"]
faces_ijk  = face_mesh["faces_ijk"].long()

# Robust face-ID lookup
points_face_id = face_mesh.get(
    "points_face_id",
    face_mesh.get("face_id", None),
)

boundary_uv = face_mesh.get(
    "boundary_curve_uv",
    face_mesh.get("boundary_uv", None),
)

boundary_face_id = face_mesh.get(
    "boundary_face_id",
    None,
)

boundary_curve_offsets = face_mesh.get(
    "boundary_curve_offsets",
    None,
)

In [8]:
test_seed_cases15 = {
    "handoff_overcomplete": random_seeds_min_dist(
        N=80,
        min_dist=0.035,
        seed=727,
        device=device,
    ),
}

In [9]:
results = []

test_seed_cases11 = {

    "normal_random": random_seeds_min_dist(
        N=50,
        min_dist=0.1,
        seed=344,
        device=device,
    ),
}


for case_name, seeds_uv in test_seed_cases15.items():

    try:

        metrics, out_p1, out_p2 = compare_phase1_phase2(

            case_name,
            seeds_uv,

            tau=0.01,

            strut_thickness=0.2,

            tube_beta=0.01,

            duplicate_merge_sigma=0.1,

            rho_threshold=0.50,

            projection_beta=8.0,
            projection_eta=0.5,
            projection_strength=1.0,

            boundary_attach_width=2e-5,
            boundary_attach_beta=1e-5,
            boundary_attach_alpha=1.0,

            # Fibre arrows
            fiber_density_threshold=0.50,
            max_fiber_arrows=1000,
            fiber_arrow_scale=0.015,

            visualize=True,
        )

        results.append(
            metrics
        )

    except Exception as exc:

        print(
            f"\nFAILED: {case_name}"
        )

        print(
            type(exc).__name__,
            str(exc),
        )


CASE: handoff_overcomplete
Seeds                 : 80
Full strut thickness  : 0.200000
Strut radius          : 0.100000

--- Nominal thickness ---
Requested full thickness : 0.2
P1 imposed radius        : 0.1
P1 implied full thickness: 0.2
P2 reported radius       : 0.10000000149011612
P2 full thickness        : 0.20000000298023224

--- Density ranges ---
P1 raw        : min=0.061844  mean=0.461882  max=1.000000
P2 raw        : min=0.000000  mean=0.361823  max=0.730862
P1 projected  : min=0.000567  mean=0.371925  max=1.000000
P2 projected  : min=0.000000  mean=0.334917  max=0.976046

--- Length ---
P1 continuous Voronoi : 144.663727
P2 internal Voronoi   : 161.183365
P2 boundary shell     : 40.000000
P2 total              : 201.183380
P1/P2 internal error  : 10.249 %

--- Raw density area sweep ---
rho >= 0.01: P1= 100.0000   P2=  96.5900   ratio=1.0353
rho >= 0.10: P1=  95.1723   P2=  81.3869   ratio=1.1694
rho >= 0.25: P1=  55.9452   P2=  61.9699   ratio=0.9028
rho >= 0.50: P1=  35.

Widget(value='<iframe src="http://localhost:39739/index.html?ui=P_0x7aec538cd690_1&reconnect=auto" class="pyvi…


--- Fibre visualization ---
P1 arrows shown: 904
P2 arrows shown: 974
Fibre density threshold: 0.5


In [10]:
a = out_p1["seed_active_weights"].detach().cpu()

print("\n--- Seed activity distribution ---")
print("min  :", float(a.min()))
print("max  :", float(a.max()))
print("mean :", float(a.mean()))

for t in [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    print(
        f"activity >= {t:4.2f}:",
        int((a >= t).sum()),
    )


--- Seed activity distribution ---
min  : 0.04280557483434677
max  : 1.0
mean : 0.2876841127872467
activity >= 0.05: 66
activity >= 0.10: 31
activity >= 0.25: 26
activity >= 0.50: 19
activity >= 0.75: 16
activity >= 0.90: 12
activity >= 0.95: 9


In [11]:
voxel_size = float((dx + dy + dz) / 120.0)
thickness = 3.0 * voxel_size
print(f"dx:{dx} ,dy:{dy} ,dz:{dz}")
print(f"voxel size : {voxel_size}")
print(f"thickness : {thickness}")
LoadingCase ="New2stage"
common_settings = dict(
    thickness=thickness,
    voxel_size=voxel_size,
    extra_layers=1,
    tensors=face_mesh,

    voxelization_mode="triangle_distance",
    subvoxel_samples=1,
    min_geom_fraction=0.25,

    faces_index_base=0,
)

shell_problem = ThickenShell(
    **common_settings
)

shell_problem.show_boundaries(
    segments_per_curve=2
)
Boundary_loads = [

    {
        "type": "force",
        "boundaries": [5],
        "magnitude": 1,
        "direction": "x",
    },
    #{
    #     "type": "distributed_force",
    #     "boundaries": [0,1],
    #     "magnitude": 5.0,
    #     "direction": "y",
    #     "distribution": "parabolic",
    # },
]


BC_Report = shell_problem.boundary_loading(
    fixed=[0,1],
    loads=Boundary_loads,
    band=0.05* voxel_size,
)
loading_img = shell_problem.show_voxels_surface_and_bc(
    return_img=True,
    off_screen=True,
    window_size=(560, 430),
    show=True,
)

dx:10.000000199999999 ,dy:10.000000199999999 ,dz:2e-07
voxel size : 0.16666667166666663
thickness : 0.5000000149999999


Widget(value='<iframe src="http://localhost:39739/index.html?ui=P_0x7ae92e7d4b80_2&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:39739/index.html?ui=P_0x7ae9ac843220_3&reconnect=auto" class="pyvi…

In [12]:
verification = VoxelFEMAbaqusRunner(
    face_mesh=face_mesh,
    shell_problem=shell_problem,
    device=device,
    output_dir="FEM_Verification",
)
LoadingCase= "bend"

/home/arash/HVD_SeedsBase/neuraltomo_fem/FE.py:31: RuntimeWarning: Legacy material keys Ef/Et/nuf/nut are mapped to explicit orthotropic CCF fields. Prefer material_E1/E2/E3, material_nu12/nu23/nu13, and material_G12/G23/G13.
  self.H8 = H8_anisotropic_K(device, element_size=self.mesh.elemSize, **problem.materialProperty)


In [13]:
from phase1_fem_gradient_diagnostic import diagnose_phase1_fem_gradients
import importlib
import phase1_fem_gradient_diagnostic

importlib.reload(phase1_fem_gradient_diagnostic)

from phase1_fem_gradient_diagnostic import diagnose_phase1_fem_gradients

In [14]:
seeds_test = random_seeds_min_dist(
    N=40,
    min_dist=0.05,
    seed=727,
    device=device,
)

report = diagnose_phase1_fem_gradients(
    face_mesh=face_mesh,
    shell_problem=shell_problem,
    seeds_uv=seeds_test,

    strut_thickness=0.40,

    tau=0.01,
    tube_beta=0.01,

    duplicate_merge_distance_factor=1.10,
    point_chunk_size=32,

    handoff_activity_threshold=0.05,
    handoff_territory_ratio=0.20,

    projection_beta=8.0,
    projection_eta=0.50,
    projection_strength=1.0,

    fem_max_displacement=20.0,
    fem_yield_strength=45000.0,

    fem_constraint_weight=2.0,
    fem_baseline_weight=0.001,

    fem_stress_density_threshold=0.50,

    # Beginning-of-training value
    fem_rho_min_ratio=1e-3,

    fem_penal=3.0,
    fem_violation_power=3.0,

    run_component_breakdown=True,
)


PHASE-1 -> FEM GRADIENT ISOLATION
device                    : cuda:0
dtype                     : torch.float32
seed count                : 40
strut thickness           : 0.4
duplicate threshold       : 0.44
point chunk size          : 32
FEM max displacement      : 20
FEM yield strength        : 45000

[full_fem] input diagnostics: rho=[6.734e-05, 1.000e+00], fiber finite=True, fiber norm=[1.000e+00, 1.000e+00]
       FEM valid=True | stress_max=2.299334e+01 | disp_max=1.750541e-02 | Rstress=5.110e-04 | Rdisp=8.753e-04
PASS | full_fem                       | loss= 1.834736e-06 | |g|= 1.389e-05 | gmax= 8.954e-06 | nan=   0 inf+=   0 inf-=   0

[density_only_fixed_fiber] input diagnostics: rho=[6.734e-05, 1.000e+00], fiber finite=True, fiber norm=[1.000e+00, 1.000e+00]
       FEM valid=True | stress_max=2.371995e+01 | disp_max=2.237048e-02 | Rstress=5.271e-04 | Rdisp=1.119e-03
PASS | density_only_fixed_fiber       | loss= 2.194298e-06 | |g|= 5.019e-06 | gmax= 2.604e-06 | nan=   0 inf+= 